# Tokenization & the token budget

**Session 1 · Foundations**

See that models operate on *tokens*, count them, and reason about the context budget.

> `count_tokens` uses a GPT-family tokenizer (tiktoken). The local Llama models this
> course runs on tokenize a bit differently, so treat every number here as an estimate
> good to ~10-20% — fine for budgeting, not for billing.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import count_tokens, TOKENIZER_NOTE
print(TOKENIZER_NOTE)

Token counts below use tiktoken (OpenAI's o200k_base). A Llama model tokenizes differently -- expect the real count to differ by ~10-20%. Good enough for budgeting.


Tokens are not words. Count a few strings:

In [2]:
for s in ["Prompt engineering", "antidisestablishmentarianism", "GPT-4o", "🙂 emoji test"]:
    print(f"{count_tokens(s):>3} tokens | {s}")

  2 tokens | Prompt engineering
  6 tokens | antidisestablishmentarianism
  4 tokens | GPT-4o
  3 tokens | 🙂 emoji test


### Budget
If the context window is 8192 tokens and your prompt uses `p`, the model has `8192 - p` left for its answer.

In [3]:
window = 8192
prompt = "You are a helpful assistant.\n" + ("blah " * 500)
p = count_tokens(prompt)
print("prompt tokens:", p, "| left for answer:", window - p)

prompt tokens: 507 | left for answer: 7685


### Worked example

A string whose token count dwarfs its word count, plus the cost math. Change the strings and the price, then re-run.


In [4]:
# Worked example: tokens vs words, and what a prompt costs
samples = [
    "supercalifragilisticexpialidocious",          # 1 word, many tokens
    "https://example.com/path?q=token&lang=en",     # punctuation explodes
    "the the the the the the the the",              # 8 words ~ 8 tokens
]
for s in samples:
    words = len(s.split())
    ratio = count_tokens(s) / max(words, 1)
    print(f"{count_tokens(s):>3} tok / {words:>2} words  ({ratio:.1f}x) | {s}")

price_per_1m = 0.15   # $ per 1M input tokens
tokens = 2000
cost = tokens / 1_000_000 * price_per_1m
print(f"\n2000-token prompt: ${cost:.6f}  (= ${cost * 1000:.3f} per 1000 calls)")


 10 tok /  1 words  (10.0x) | supercalifragilisticexpialidocious
 11 tok /  1 words  (11.0x) | https://example.com/path?q=token&lang=en
  8 tok /  8 words  (1.0x) | the the the the the the the the

2000-token prompt: $0.000300  (= $0.300 per 1000 calls)


## Your turn - vary the example

1. Beat the example: find a string with an even higher token/word ratio.
2. Re-price the 2000-token prompt at a frontier rate (e.g. $3 / 1M) and at 100k calls/day.
3. Add one sample of your own and predict its token count before running.
4. Count the same string with `count_tokens(s, model="gpt-4")` vs the default. How much does the
   tokenizer choice alone move the number? That gap is your error bar.


In [5]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
